In [1]:
import math
#5x5
SIZE = 5
WIN_CONDITION = 4
board = [[' ' for _ in range(SIZE)] for _ in range(SIZE)]

def check_winner(b):

    for r in range(SIZE):
        for c in range(SIZE):
            if b[r][c] == ' ': continue

            for dr, dc in [(0,1), (1,0), (1,1), (1,-1)]:
                count = 0
                for i in range(WIN_CONDITION):
                    nr, nc = r + dr*i, c + dc*i
                    if 0 <= nr < SIZE and 0 <= nc < SIZE and b[nr][nc] == b[r][c]:
                        count += 1
                    else: break
                if count == WIN_CONDITION: return b[r][c]

    if all(cell != ' ' for row in b for cell in row): return 'Tie'
    return None

def evaluate_board(b):
    """ Hàm đánh giá thế trận khi chưa kết thúc game """

    return 0

def minimax(board, depth, alpha, beta, is_maximizing):
    result = check_winner(board)
    if result == 'X': return 100 - depth
    if result == 'O': return depth - 100
    if result == 'Tie': return 0

    if depth >= 4:
        return evaluate_board(board)

    if is_maximizing:
        best_score = -math.inf
        for r in range(SIZE):
            for c in range(SIZE):
                if board[r][c] == ' ':
                    board[r][c] = 'X'
                    score = minimax(board, depth + 1, alpha, beta, False)
                    board[r][c] = ' '
                    best_score = max(score, best_score)
                    alpha = max(alpha, score)
                    if beta <= alpha: break
        return best_score
    else:
        best_score = math.inf
        for r in range(SIZE):
            for c in range(SIZE):
                if board[r][c] == ' ':
                    board[r][c] = 'O'
                    score = minimax(board, depth + 1, alpha, beta, True)
                    board[r][c] = ' '
                    best_score = min(score, best_score)
                    beta = min(beta, score)
                    if beta <= alpha: break
        return best_score

def find_best_move(board):
    best_val = -math.inf
    move = (-1, -1)

    for r in range(SIZE):
        for c in range(SIZE):
            if board[r][c] == ' ':
                board[r][c] = 'X'
                move_val = minimax(board, 0, -math.inf, math.inf, False)
                board[r][c] = ' '
                if move_val > best_val:
                    best_val = move_val
                    move = (r, c)
    return move

def print_board(b):
    print("\n   " + "   ".join(map(str, range(SIZE))))
    for i, row in enumerate(b):
        print(f"{i} " + " | ".join(row))
        if i < SIZE - 1: print("  " + "---+" * (SIZE-1) + "---")

# --- Game Loop ---
print(f"TIC-TAC-TOE {SIZE}x{SIZE} (Cần {WIN_CONDITION} ô để thắng)")
while True:
    print_board(board)
    res = check_winner(board)
    if res: break

    # Lượt người
    while True:
        try:
            r, c = map(int, input(f"Nhập hàng cột (0-{SIZE-1}): ").split())
            if board[r][c] == ' ':
                board[r][c] = 'O'
                break
        except: print("Nhập sai!")

    if check_winner(board): break


    print("AI đang tính toán...")
    m = find_best_move(board)
    if m != (-1, -1): board[m[0]][m[1]] = 'X'

print_board(board)
final = check_winner(board)
print(f"KẾT THÚC: {final if final != 'Tie' else 'Hòa'}")

TIC-TAC-TOE 5x5 (Cần 4 ô để thắng)

   0   1   2   3   4
0   |   |   |   |  
  ---+---+---+---+---
1   |   |   |   |  
  ---+---+---+---+---
2   |   |   |   |  
  ---+---+---+---+---
3   |   |   |   |  
  ---+---+---+---+---
4   |   |   |   |  
Nhập hàng cột (0-4): 0 4
AI đang tính toán...

   0   1   2   3   4
0 X |   |   |   | O
  ---+---+---+---+---
1   |   |   |   |  
  ---+---+---+---+---
2   |   |   |   |  
  ---+---+---+---+---
3   |   |   |   |  
  ---+---+---+---+---
4   |   |   |   |  
Nhập hàng cột (0-4): 3 3
AI đang tính toán...

   0   1   2   3   4
0 X | X |   |   | O
  ---+---+---+---+---
1   |   |   |   |  
  ---+---+---+---+---
2   |   |   |   |  
  ---+---+---+---+---
3   |   |   | O |  
  ---+---+---+---+---
4   |   |   |   |  
Nhập hàng cột (0-4): 2 2
AI đang tính toán...

   0   1   2   3   4
0 X | X | X |   | O
  ---+---+---+---+---
1   |   |   |   |  
  ---+---+---+---+---
2   |   | O |   |  
  ---+---+---+---+---
3   |   |   | O |  
  ---+---+---+---+---
4   |  

In [2]:
import math
#10x10
SIZE = 10
WIN_CON = 5
DEPTH_LIMIT = 2

board = [[' ' for _ in range(SIZE)] for _ in range(SIZE)]

def evaluate_line(line, player):
    score = 0
    opp = 'O' if player == 'X' else 'X'

    count = line.count(player)
    empty = line.count(' ')

    if count == 5: score += 100000
    elif count == 4 and empty == 1: score += 10000
    elif count == 3 and empty == 2: score += 1000
    elif count == 2 and empty == 3: score += 100
    return score

def heuristic_evaluation(b):
    score = 0

    for r in range(SIZE):
        for c in range(SIZE - WIN_CON + 1):
            window = b[r][c:c+WIN_CON]
            score += evaluate_line(window, 'X')
            score -= evaluate_line(window, 'O')
    return score

def get_nearby_moves(b):
    """ Chỉ lấy các ô trống xung quanh các quân đã đánh để tăng tốc """
    moves = set()
    for r in range(SIZE):
        for c in range(SIZE):
            if b[r][c] != ' ':
                for dr in range(-1, 2):
                    for dc in range(-1, 2):
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < SIZE and 0 <= nc < SIZE and b[nr][nc] == ' ':
                            moves.add((nr, nc))
    return list(moves) if moves else [(SIZE//2, SIZE//2)]

def minimax(board, depth, alpha, beta, is_max):
    if depth == DEPTH_LIMIT:
        return heuristic_evaluation(board)

    moves = get_nearby_moves(board)
    if is_max:
        best = -math.inf
        for r, c in moves:
            board[r][c] = 'X'
            score = minimax(board, depth + 1, alpha, beta, False)
            board[r][c] = ' '
            best = max(best, score)
            alpha = max(alpha, best)
            if beta <= alpha: break
        return best
    else:
        best = math.inf
        for r, c in moves:
            board[r][c] = 'O'
            score = minimax(board, depth + 1, alpha, beta, True)
            board[r][c] = ' '
            best = min(best, score)
            beta = min(beta, best)
            if beta <= alpha: break
        return best

def find_best_move(b):
    best_val = -math.inf
    move = None
    for r, c in get_nearby_moves(b):
        b[r][c] = 'X'
        val = minimax(b, 0, -math.inf, math.inf, False)
        b[r][c] = ' '
        if val > best_val:
            best_val = val
            move = (r, c)
    return move

def print_board(b):
    print("   " + " ".join(f"{i:2}" for i in range(SIZE)))
    for i, row in enumerate(b):
        print(f"{i:2} " + " | ".join(row))

# Game Loop đơn giản
print("Tic-Tac-Toe 10x10 (Cần 5 ô để thắng)")
while True:
    print_board(board)
    try:
        r, c = map(int, input("Lượt bạn (hàng cột): ").split())
        if board[r][c] == ' ': board[r][c] = 'O'
        else: continue
    except: break

    print("AI đang tính...")
    m = find_best_move(board)
    if m: board[m[0]][m[1]] = 'X'

Tic-Tac-Toe 10x10 (Cần 5 ô để thắng)
    0  1  2  3  4  5  6  7  8  9
 0   |   |   |   |   |   |   |   |   |  
 1   |   |   |   |   |   |   |   |   |  
 2   |   |   |   |   |   |   |   |   |  
 3   |   |   |   |   |   |   |   |   |  
 4   |   |   |   |   |   |   |   |   |  
 5   |   |   |   |   |   |   |   |   |  
 6   |   |   |   |   |   |   |   |   |  
 7   |   |   |   |   |   |   |   |   |  
 8   |   |   |   |   |   |   |   |   |  
 9   |   |   |   |   |   |   |   |   |  
Lượt bạn (hàng cột): 0 0
AI đang tính...
    0  1  2  3  4  5  6  7  8  9
 0 O | X |   |   |   |   |   |   |   |  
 1   |   |   |   |   |   |   |   |   |  
 2   |   |   |   |   |   |   |   |   |  
 3   |   |   |   |   |   |   |   |   |  
 4   |   |   |   |   |   |   |   |   |  
 5   |   |   |   |   |   |   |   |   |  
 6   |   |   |   |   |   |   |   |   |  
 7   |   |   |   |   |   |   |   |   |  
 8   |   |   |   |   |   |   |   |   |  
 9   |   |   |   |   |   |   |   |   |  
Lượt bạn (hàng cột): 0 1
    0  1  2